# 1. First principles: what a CNN costs, and how we will climb

This folder builds a EuroSAT classifier **from a naive CNN**, one measured step at a time,
to beat the leaderboard's leader before it, BuzinD, on all three numbers at once:

| | test accuracy | training compute | compute per image |
|---|---|---|---|
| BuzinD | 96.57% | 26.2 TFLOP | 119 MFLOP |

Nothing is borrowed from BuzinD's code or from the earlier demos: every choice below is
made by an experiment that asks one question. The techniques are the generic ones you
will use in practice (batch normalization, learning-rate schedules, augmentation,
residual blocks, choosing where to spend compute), not tricks.

This notebook sets up the tools:

1. **The bar on our own images.** What BuzinD scores on the held-back images we will use.
2. **What a convolution costs**, worked out by hand, and checked against the counter the
   leaderboard uses.
3. **Where the naive CNN spends its FLOPs**, layer by layer.
4. **A surprise in the counter** that changes which designs pay off on this leaderboard.
5. **The budget** we have to stay inside, and **the method**: one question per
   experiment, 3 seeds, a standard error, and a split kept aside to confirm.

Every experiment lives in `experiments.py` and ran on a rented GPU (see `pod/README.md`).
The notebooks only read the saved runs in `runs/`, so they run in seconds on a laptop.

In [1]:
import torch
from analyze import layer_flops, show
from search import Config, count_flops

torch.set_printoptions(precision=3)

## 1. The bar, on our images

`uv run score` holds back 20% of the training images to check a model (split 7). To
choose settings without fooling ourselves, every choice in this folder is made on a
*different* split (11), and the final choice is confirmed on splits 7 and 3, which played
no part in making it.

BuzinD's own `model.py`, scored on this computer (`uv run score submissions/BuzinD
--split-seed 11`, then 7 and 3), one run each:

| split | 11 (choose) | 7 (confirm) | 3 (confirm) |
|---|---|---|---|
| BuzinD | 96.67% | 97.06% | 96.81% |

It trained in 184 s here and 407 s on GitHub's computer: GitHub is about 2.2 times slower,
so whatever we submit has to train in well under 270 s here to fit GitHub's 10 minutes.

## 2. What a convolution costs

A 3x3 convolution from `C_in` to `C_out` channels on an `H x W` image computes, for each
of the `H x W x C_out` outputs, a weighted sum of `3 x 3 x C_in` inputs: one multiply and
one add each. So its forward pass costs

$$\text{FLOPs} = 2 \cdot H \cdot W \cdot C_{out} \cdot 9 \cdot C_{in}$$

Three consequences drive every design decision later:

- **Doubling the channels quadruples the cost** (both `C_in` and `C_out` double).
- **Halving the image quarters the cost.** So the classic pattern, *halve the image and
  double the channels*, costs the same in every stage.
- **Training costs about 3 forward passes**: the backward pass computes a gradient for
  the input and one for the weights, each as costly as the forward pass (the first
  layer needs no input gradient, so a little less than 3).

Let us check the formula against the counter the leaderboard uses (`eurosat.flops`).

In [2]:
from torch import nn

from eurosat.flops import counting

conv = nn.Conv2d(32, 64, 3, padding=1)
x = torch.rand(1, 32, 16, 16, requires_grad=True)
with counting() as forward:
    y = conv(x)
with counting() as backward:
    y.sum().backward()
print(f"by hand:  {2 * 16 * 16 * 64 * 9 * 32:,}")
print(
    f"counted:  {forward.flops:,} forward, {backward.flops:,} backward "
    f"({backward.flops / forward.flops:.0f}x)"
)

by hand:  9,437,184
counted:  9,437,184 forward, 18,874,368 backward (2x)


## 3. Where the naive CNN spends its FLOPs

The starting point is the CNN everyone writes first: three 3x3 convolutions (16, 32, 64
channels), each followed by ReLU and a 2x2 max pool, then flatten and one linear layer.
In this folder's notation (see `search.py`): `"c16 p c32 p c64 p fl"`.

In [3]:
naive = Config()
print(naive.arch)
for word, mflop in layer_flops(naive):
    if mflop:
        print(f"  {word:>5}: {mflop:5.2f} MFLOP")
print(count_flops(naive, n_train=17280))

c16 p c32 p c64 p fl
    c16:  3.54 MFLOP
    c32:  9.44 MFLOP
    c64:  9.44 MFLOP
   head:  0.08 MFLOP
{'train_flops': 11049998745600, 'predict_flops': 22495232, 'params': 64554}


It costs 22.5 MFLOP to look at one image. The first layer works on the full 64x64
image but has only 3 input channels, so it is the cheapest; the second and third cost the
same 9.4 MFLOP (half the image, twice the channels). The linear layer at the end is
almost free in FLOPs but holds 41,000 of the 65,000 parameters.

Training it for 10 epochs on 17,280 images costs 11 TFLOP: already 40% of BuzinD's whole
budget, for a network that will turn out to score 79%. Compute has to be spent with care.

## 4. A surprise in the counter

A *depthwise-separable* convolution (MobileNet) replaces one 3x3 convolution by a 3x3
convolution *per channel* (`groups = C`) and a 1x1 convolution that mixes the channels.
On paper it is about 8 times cheaper. Look at how the counter sees its backward pass:

In [4]:
for groups in (1, 8, 64):
    conv = nn.Conv2d(64, 64, 3, padding=1, groups=groups, bias=False)
    x = torch.rand(2, 64, 16, 16, requires_grad=True)
    with counting() as forward:
        y = conv(x)
    with counting() as backward:
        y.sum().backward()
    print(
        f"groups {groups:>2}: forward {forward.flops:>10,}   backward {backward.flops:>10,}"
        f"  = {backward.flops / forward.flops:4.0f}x forward (the real work: 2x)"
    )

groups  1: forward 37,748,736   backward 75,497,472  =    2x forward (the real work: 2x)
groups  8: forward  4,718,592   backward 42,467,328  =    9x forward (the real work: 2x)
groups 64: forward    589,824   backward 38,338,560  =   65x forward (the real work: 2x)


PyTorch's FLOP counter, which the leaderboard uses, counts the backward pass of a
grouped convolution as if it were an ordinary (dense) one: 65 times the forward pass for
a depthwise convolution instead of 2. The forward count, and so the compute per image, is
right; the *training* compute of any network with depthwise convolutions is counted many
times too high.

We do not change the counter: it is the leaderboard's ruler, and it is the same for
everyone. But it decides strategy: on this leaderboard, depthwise-separable and
inverted-residual blocks are charged far more than they cost, so the climb keeps to
ordinary and residual convolutions, which are counted exactly. (Experiment 7 measures
what that costs us.) The general lesson: **know how your metric is measured** before you
optimize for it.

## 5. The budget, and the method

Training compute is about `2.95 x (forward FLOPs per image) x 21,600 images x epochs`.
BuzinD's network costs 14.9 MFLOP per image and trains 28 epochs: about **420
MFLOP-epochs**, 26.2 TFLOP. To beat it on training compute, *forward MFLOP x epochs* has
to stay below that: 10 MFLOP for 40 epochs, or 20 for 20. It answers with 8 turned views
of each image (119 MFLOP), so a network of 15 MFLOP can afford up to 7 views.

**The method**, used for every experiment in `2-climb.ipynb`:

1. **One question**, written down with a **prediction** before the runs.
2. **3 seeds per setting.** We report the mean and its *standard error* (the spread of the
   seeds divided by the square root of 3): two settings closer than about two standard
   errors may differ by luck alone.
3. **Look at the curves, not just the score.** Training accuracy far above check
   accuracy means memorizing (regularize, augment); the two equal means underfitting
   (more capacity, more or better-spent compute).
4. **Compare at equal compute** when a change costs FLOPs.
5. **Choose on split 11, confirm on splits 7 and 3.**

The runs are in `runs/`. For example, the first experiment:

In [5]:
show("0-naive")

experiment    setting                            n    check acc  1 view  train  TFLOP  MFLOP   params  gpu s
0-naive       naive 10 ep                        3  78.65 ±1.93   78.65  83.60  11.05   22.5    64554     49
0-naive       naive 30 ep                        3  84.34 ±0.18   84.34  93.99  33.15   22.5    64554    118
